# HW5. Olist: CTE-chain, recursive CTE, Materialized View, EXPLAIN ANALYZE

Дані: Brazilian E-Commerce (Olist). СУБД: локальний PostgreSQL 16 (pgAdmin), база hw5.

## Завдання 1. Setup і typed-схема Olist

In [1]:
%pip install -q psycopg2-binary sqlalchemy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import zipfile
import getpass
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Пароль не зберігається у файлі: береться зі змінної середовища або вводиться вручну.
DB_URL = os.getenv("HW5_DB_URL")
if not DB_URL:
    DB_URL = f"postgresql+psycopg2://postgres:{getpass.getpass('Пароль користувача postgres: ')}@localhost:5432/hw5"

engine = create_engine(DB_URL, future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


def run(sql_text):
    """Виконує SQL (можна кілька команд). Якщо остання повертає рядки, віддає DataFrame."""
    with engine.begin() as conn:
        res = conn.exec_driver_sql(sql_text)
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))


def show_plan(sql_text):
    """Друкує output EXPLAIN ANALYZE як звичайний текст (щоб він зберігся у notebook)."""
    plan = run(sql_text)
    print("\n".join(plan.iloc[:, 0]))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd4 in position 61: invalid continuation byte

In [ ]:
# CSV Olist взято з архіву Kaggle (archive.zip) і розпаковано локально.
ZIP_PATH = r"C:\Users\PC\Downloads\archive.zip"
EXTRACT_DIR = r"C:\Users\PC\Downloads\olist_data"

dataset_dir = os.getenv("OLIST_DIR")
if not dataset_dir:
    if not os.path.isdir(EXTRACT_DIR):
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(EXTRACT_DIR)
    # шукаємо папку, де лежить olist_customers_dataset.csv (може бути вкладена)
    for root, _, names in os.walk(EXTRACT_DIR):
        if "olist_customers_dataset.csv" in names:
            dataset_dir = root
            break

print(dataset_dir)

In [ ]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
}

raw_counts = {}
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)
    raw_counts[table_name] = len(df)
    print(table_name, df.shape)

Raw-таблиці лише проміжний шар (копія CSV). Усі наступні завдання працюють з typed-таблицями, у яких є типи та constraints.

In [ ]:
run("""
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
""")

In [ ]:
run("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;
""")

У вихідному CSV колонки products названо product_name_lenght та product_description_lenght (з помилкою). У typed-схемі використано нормалізовані назви product_name_length і product_description_length, а в SELECT із raw-таблиці фактичні назви з CSV.

In [ ]:
run("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
""")

In [ ]:
# порівняння з raw-таблицями (кількість рядків має збігатися)
typed = run("""
SELECT 'olist_customers_raw' AS raw_table, COUNT(*) AS typed_rows FROM olist_customers
UNION ALL SELECT 'olist_orders_raw', COUNT(*) FROM olist_orders
UNION ALL SELECT 'olist_order_items_raw', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'olist_products_raw', COUNT(*) FROM olist_products
UNION ALL SELECT 'olist_sellers_raw', COUNT(*) FROM olist_sellers
""")
typed["raw_rows"] = typed["raw_table"].map(raw_counts)
typed["diff"] = typed["raw_rows"] - typed["typed_rows"]
typed

Висновок smoke-test: кількість рядків у кожній із п'яти typed-таблиць збігається з raw (diff = 0), тож під час перенесення нічого не втрачено: усі значення пройшли приведення типів і constraints, зокрема CHECK на order_status.

## Завдання 2. CTE-chain pipeline з щонайменше п'ятьма stages

Pipeline будує customer-level ознаки для ML: RFM, cohort_quarter, avg_order_value_30d, cancellation_rate і preferred_category. Кожен stage виконує одну логічну трансформацію і має коментар Stage N.

In [ ]:
run("""
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;
""")

Структура pipeline: кожен stage виконує одну логічну трансформацію, тож його можна перевірити окремо.

Stage 0: reference_date береться з самого датасету (MAX(order_purchase_timestamp) + 1 день), а не з CURRENT_DATE, тому recency_days відтворюваний. Stage 1: сума кожного замовлення (ціна плюс доставка). Stage 2: RFM (recency_days, frequency, monetary) і first_order_date. Stage 3: cohort_quarter через DATE_TRUNC('quarter', ...). Stage 4: rolling 30-day AVG order_total через window function з RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW і ROW_NUMBER для вибору останнього замовлення. Stage 5: cancellation_rate як середнє від CASE WHEN order_status = 'canceled'. Stage 6: revenue за парою клієнт і категорія. Stage 7: preferred_category як top-1 за revenue через ROW_NUMBER. Фінальний SELECT збирає все через LEFT JOIN від olist_customers, а COALESCE дає значення за замовчуванням клієнтам без замовлень.

Результат: найбільший monetary має клієнт 1617b1357756262bfa56ab541c47bc16 зі штату RJ (13 664,08, одне замовлення, preferred_category = telefonia_fixa, cohort_quarter = 2017-07-01).

In [ ]:
# перевірка логіки окремих stages: розподіл frequency, відсутні замовлення, cohort-квартали
run("""
WITH orders_per_customer AS (
    SELECT c.customer_id, COUNT(o.order_id) AS n_orders
    FROM olist_customers AS c
    LEFT JOIN olist_orders AS o
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_id
)
SELECT n_orders, COUNT(*) AS n_customers
FROM orders_per_customer
GROUP BY n_orders
ORDER BY n_orders;
""")

Усі 99 441 customer_id мають рівно одне замовлення, тому frequency у pipeline завжди дорівнює 1, cancellation_rate може бути лише 0 або 1, а avg_order_value_30d збігається із сумою єдиного замовлення. Це наслідок того, що в Olist customer_id створюється окремо для кожного замовлення. За умовою завдання pipeline працює на рівні customer_id, але для реальних ML-ознак повторних покупок потрібно групувати за customer_unique_id (як у HW4).

## Завдання 3. Recursive CTE для referral chain

Olist не містить referral-таблиці, тому створюється синтетична hw5_referrals: десять перших клієнтів (за customer_id) утворюють ланцюжок, де кожного наступного запросив попередній.

In [ ]:
run("""
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
""")

In [ ]:
run("""
SELECT *
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
""")

In [ ]:
run("""
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
""")

Пояснення: anchor part вибирає root-клієнтів, яких ніхто не запрошував (NOT EXISTS замість NOT IN, щоб NULL у підзапиті не зламав результат). Recursive part переходить від referrer до його referred-клієнтів і накопичує depth, path_ids та path_text.

Результат: на depth = 1 стоять 99 432 клієнти (99 441 мінус 9 запрошених), а на depth від 2 до 10 по одному клієнту. Це ланцюжок із 10 клієнтів, де глибина останнього дорівнює 10.

Termination: рекурсія зупиняється, коли recursive part не повертає нових рядків. Додатково стоять два захисти від циклів: NOT r.referred_customer_id = ANY(rd.path_ids) та depth < 50.

Демонстрація захисту від циклу A → B → A на вбудованих даних (без зміни таблиці hw5_referrals): рекурсія зупиняється, бо B → A відсікається перевіркою NOT next_id = ANY(path_ids).

In [ ]:
run("""
WITH RECURSIVE edges(referrer_id, referred_id) AS (
    VALUES ('A', 'B'), ('B', 'A')
),
walk AS (
    SELECT
        'A'::TEXT AS node_id,
        1 AS depth,
        ARRAY['A']::TEXT[] AS path_ids
    UNION ALL
    SELECT
        e.referred_id,
        w.depth + 1,
        w.path_ids || e.referred_id
    FROM edges AS e
    JOIN walk AS w
        ON w.node_id = e.referrer_id
    WHERE NOT e.referred_id = ANY(w.path_ids)
      AND w.depth < 50
)
SELECT node_id, depth, path_ids
FROM walk
ORDER BY depth;
""")

Результат: вузол A на depth = 1 і вузол B на depth = 2. Ребро B → A відсікається, бо A уже є в path_ids, тож рекурсія завершується. Без перевірки ANY(path_ids) цикл A → B → A → B ... не мав би природного кінця, і зупинив би його лише depth guard після 50 кроків зі зайвими дублікатами. У PostgreSQL для цього також є CYCLE-clause.

## Завдання 4. Materialized View з CONCURRENT REFRESH

MV містить повний результат CTE-pipeline із Завдання 2 (ті самі stages і ті самі колонки), без LIMIT.

In [ ]:
run("""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;
""")

In [ ]:
run("""
CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);
""")

In [ ]:
run("""
SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;
""")

In [ ]:
run("""
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
""")

Concurrent refresh виконується окремою клітинкою після створення UNIQUE INDEX.

In [ ]:
run("""
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;
""")

In [ ]:
# перевірка, що MV і далі містить усі рядки та що індекс унікальний
run("""
SELECT
    (SELECT COUNT(*) FROM mv_olist_customer_features) AS mv_rows,
    (SELECT COUNT(*) FROM olist_customers)            AS customers_rows,
    (SELECT indexdef FROM pg_indexes
      WHERE indexname = 'idx_mv_olist_customer_features_customer_id') AS index_definition;
""")

Пояснення: Materialized View містить повний результат pipeline (99 441 рядок, стільки ж, скільки клієнтів), ті самі колонки й те саме топ-5, що в Завданні 2. UNIQUE INDEX на customer_id потрібен для двох речей: швидкого lookup і команди REFRESH MATERIALIZED VIEW CONCURRENTLY. Вона працює лише за наявності UNIQUE INDEX і не блокує читання під час оновлення, на відміну від звичайного REFRESH, який бере ексклюзивне блокування.

## Завдання 5. EXPLAIN ANALYZE і Reflection

### 5.1. EXPLAIN на повному CTE-pipeline (stages 0 до 2, топ-10 за monetary)

In [ ]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
)
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")

### 5.2. EXPLAIN на lookup із Materialized View за customer_id

In [ ]:
# оновлюємо статистику MV, щоб планувальник бачив реальну кількість рядків
run("ANALYZE mv_olist_customer_features;")

show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;
""")

### 5.3. Top-N читання з MV з окремим індексом на monetary

In [ ]:
run("""
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);
""")
run("ANALYZE mv_olist_customer_features;")

show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")

### 5.4. Індекс, що відповідає порядку ORDER BY (monetary DESC NULLS LAST)

In [ ]:
run("""
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary_nl
    ON mv_olist_customer_features (monetary DESC NULLS LAST);
""")
run("ANALYZE mv_olist_customer_features;")

show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")

Інтерпретація планів. Значення нижче з мого запуску, у вашому output вони будуть трохи іншими.

5.1. Повний pipeline (stages 0 до 2): Execution Time приблизно 500-700 мс. У плані видно Seq Scan по olist_orders і olist_order_items, Parallel Hash Right Join, Sort із Sort Method: external (запис на диск 6-7 МБ), Partial і Finalize GroupAggregate по order_id, HashAggregate по customer_id (Batches: 21, Disk Usage близько 7,4 МБ) і фінальний top-N heapsort. Найдорожче не JOIN (близько 100 мс), а агрегації й сортування, які не вмістилися у work_mem.

5.2. Lookup із MV: Nested Loop з Index Scan using idx_mv_olist_customer_features_customer_id, Execution Time близько 0,1 мс, тобто у тисячі разів швидше за повне переобчислення.

5.3. Top-N з MV з індексом monetary DESC: у плані Parallel Seq Scan і Sort (top-N heapsort), приблизно 26-34 мс. Індекс не використано: DESC-індекс за замовчуванням має NULLS FIRST, а запит просить NULLS LAST, тож порядок індексу не збігається з ORDER BY.

5.4. Після індексу monetary DESC NULLS LAST: Index Scan using idx_mv_olist_customer_features_monetary_nl, Execution Time близько 0,06 мс і без окремого Sort.

## Reflection

1. Найдорожчими в EXPLAIN ANALYZE виявилися агрегації й сортування, а не сам JOIN. Parallel Hash Right Join між olist_orders і olist_order_items займає близько 100 мс із приблизно 500-700 мс усього запиту, тоді як Sort по order_id (external, запис на диск 6-7 МБ) і HashAggregate по customer_id (Batches: 21, Disk Usage близько 7,4 МБ) не вмістилися у work_mem. Window functions зі stages 3-7 в цей EXPLAIN не входили, тож повний pipeline коштує ще дорожче.

2. Кешувати варто: lookup із Materialized View виконується приблизно за 0,1 мс через Index Scan, тобто у тисячі разів швидше за переобчислення. Refresh-cycle: щодоби вночі, бо ознаки залежать від історичних замовлень і хвилинна свіжість не потрібна; щогодини було б зайвим recompute-cost. Після великого batch-завантаження можна запускати REFRESH CONCURRENTLY on-demand.

3. Звичайний VIEW правильніший, коли потрібні завжди актуальні дані (наприклад, live-статус замовлення) або запит дешевий і добре індексований, тож зберігати копію немає сенсу.

4. Recursive CTE завершується, коли recursive part не повертає нових рядків: ланцюжок із 10 клієнтів зупинився на depth = 10. Якби граф містив цикл A → B → A, без захисту рекурсія була б нескінченною. Тому використано path_ids з умовою NOT next_id = ANY(path_ids), яка відсікає повернення до A, і depth guard depth < 50. Також можна застосувати CYCLE-clause.

5. Найважливіший trade-off тут recompute-cost проти freshness: ми свідомо платимо застарілістю до наступного refresh, щоб отримати latency близько 0,1 мс замість 500+ мс. Storage-cost невеликий (99 441 рядок). Окремий урок: індекс monetary DESC не спрацював через NULLS LAST, тож індекс має збігатися з порядком у запиті.

## Підсумки

У роботі створено typed-схему, побудовано CTE-chain pipeline із сімома stages для customer-level ознак, реалізовано recursive CTE із захистом від циклів, створено Materialized View з UNIQUE INDEX і concurrent refresh та порівняно плани виконання через EXPLAIN (ANALYZE, BUFFERS).